# OpenPlaque RCA automatic localization — v4 slice-wise aorta tracker

This version deliberately avoids 3-D blood-pool connectivity. It tracks the ascending aorta slice-by-slice as a large round structure, bootstrapped from the patient-right member of the aorta/pulmonary great-vessel pair in true LPS coordinates. It then searches only the right-anterior aortic wall for small persistent contrast-filled branches.

There are no sliders, widgets, clicks, or coordinates to enter. Use **Runtime → Run all**. Google Drive mounts first.


In [ ]:
# ALWAYS FIRST
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

SRC=Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
from openplaque.study import OpenPlaqueStudy
from openplaque.rca_aorta_tracker_v4 import track_ascending_aorta, find_rca_wall_candidates
print('Dependencies ready.')


In [ ]:
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP=ROOT/'Full_DICOM.zip'
LOCAL_ZIP=Path('/content/Full_DICOM.zip')
EXTRACT_ROOT='/content/full_dicom_rca_v4'
if not DRIVE_ZIP.exists(): raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print('Copying ZIP to local Colab storage...',flush=True); shutil.copyfile(DRIVE_ZIP,LOCAL_ZIP)
shutil.rmtree(EXTRACT_ROOT,ignore_errors=True)
print('Loading source series 7...',flush=True)
study=OpenPlaqueStudy(str(LOCAL_ZIP),extract_root=EXTRACT_ROOT)
source_img,source,source_files=study.load_series(7)
print('shape zyx:',source.shape,' spacing xyz:',source_img.GetSpacing(),' direction:',source_img.GetDirection())


In [ ]:
print('Tracking ascending aorta slice-by-slice...',flush=True)
t=time.time()
aorta_track=track_ascending_aorta(source,source_img)
print(f'Aorta track: {len(aorta_track)} slices, z={aorta_track[0].z}..{aorta_track[-1].z}, computed in {time.time()-t:.1f}s')
track_df=pd.DataFrame([dict(z=a.z,x=a.x,y=a.y,radius_mm=a.radius_mm,area_mm2=a.area_mm2,mean_HU=a.mean_hu,LPS_x=a.lps_x_mm,LPS_y=a.lps_y_mm,confidence=a.confidence) for a in aorta_track])
display(track_df.iloc[::max(1,len(track_df)//20)].reset_index(drop=True))


## 1. Validate the tracked ascending aorta
The black circle must stay on the large round **ascending aorta**, not the pulmonary artery or a cardiac chamber.


In [ ]:
OUT=ROOT/'RCA_AortaTrack_v4'; OUT.mkdir(parents=True,exist_ok=True)
idx=np.linspace(0,len(aorta_track)-1,12).round().astype(int)
fig,axes=plt.subplots(3,4,figsize=(16,12))
sx,sy,_=source_img.GetSpacing()
for ax,i in zip(axes.ravel(),idx):
    a=aorta_track[int(i)]
    ax.imshow(source[a.z],cmap='gray',vmin=-200,vmax=800)
    rp=a.radius_mm/np.sqrt(sx*sy)
    ax.add_patch(Circle((a.x,a.y),rp,fill=False,linewidth=1.4))
    ax.scatter([a.x],[a.y],s=18)
    ax.set_title(f'z={a.z}  r={a.radius_mm:.1f}mm  LPSx={a.lps_x_mm:.1f}')
    ax.axis('off')
plt.tight_layout()
p=OUT/'RCA_v4_aorta_track_validation.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved',p)


In [ ]:
print('Searching the right-anterior aortic wall for persistent small vessels...',flush=True)
t=time.time()
cands=find_rca_wall_candidates(source,source_img,aorta_track,n=8)
print(f'Found {len(cands)} distinct candidates in {time.time()-t:.1f}s')
cand_df=pd.DataFrame([dict(candidate=f'R{i+1}',z=c.z,x=c.x,y=c.y,score=c.score,HU=c.hu,vesselness=c.vesselness,radius_mm=c.radius_mm,distance_from_aorta_mm=c.distance_from_aorta_mm,support=c.support_slices) for i,c in enumerate(cands)])
display(cand_df)


## 2. Candidate full-context and close-up gallery
Each cyan marker is an automatically discovered small-vessel hypothesis; the black circle is the tracked ascending aorta for that slice.


In [ ]:
if cands:
    fig,axes=plt.subplots(len(cands),2,figsize=(12,5*len(cands)))
    if len(cands)==1: axes=np.asarray([axes])
    for i,(axrow,c) in enumerate(zip(axes,cands)):
        ax0,ax1=axrow
        ax0.imshow(source[c.z],cmap='gray',vmin=-200,vmax=800)
        rp=c.aorta_radius_mm/np.sqrt(sx*sy)
        ax0.add_patch(Circle((c.aorta_x,c.aorta_y),rp,fill=False,linewidth=1.4))
        ax0.scatter([c.x],[c.y],s=55,facecolors='none',linewidths=1.8)
        ax0.set_title(f'R{i+1} full context z={c.z}')
        ax0.axis('off')
        r=70; y0,y1=max(0,c.y-r),min(source.shape[1],c.y+r+1); x0,x1=max(0,c.x-r),min(source.shape[2],c.x+r+1)
        ax1.imshow(source[c.z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=800)
        ax1.scatter([c.x-x0],[c.y-y0],s=55,facecolors='none',linewidths=1.8)
        ax1.set_title(f'R{i+1} close-up HU={c.hu:.0f} support={c.support_slices} dA={c.distance_from_aorta_mm:.1f}mm')
        ax1.axis('off')
    plt.tight_layout()
    p=OUT/'RCA_v4_candidates.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
    print('Saved',p)
else:
    print('No RCA wall candidates found.')


## 3. Dense sequence around the best candidate
This is the highest-information review: it shows whether the putative vessel actually emerges from the aortic wall and persists over several millimeters.


In [ ]:
if cands:
    c=cands[0]; dz=max(1,int(round(1.2/source_img.GetSpacing()[2])))
    zs=np.arange(c.z-5*dz,c.z+6*dz,dz,dtype=int); zs=zs[(zs>=0)&(zs<source.shape[0])]
    fig,axes=plt.subplots(3,4,figsize=(16,12))
    for ax in axes.ravel(): ax.axis('off')
    r=95
    for ax,z in zip(axes.ravel(),zs):
        y0,y1=max(0,c.y-r),min(source.shape[1],c.y+r+1); x0,x1=max(0,c.x-r),min(source.shape[2],c.x+r+1)
        ax.imshow(source[z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=800)
        ax.set_title(f'z={z} ({(z-c.z)*source_img.GetSpacing()[2]:+.1f} mm from R1)')
        ax.axis('off')
    plt.tight_layout()
    p=OUT/'RCA_v4_R1_dense_sequence.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
    print('Saved',p)
print('V4 COMPLETE. Upload the aorta-track validation, candidate gallery, and dense R1 sequence.')
